# FloodWatch KP: Model B Training & Evaluation (Snowmelt & Rain-on-Snow Risk)
**Mechanism:** Snowpack Melt and Rain-on-Snow Floods  
**Architecture:** Single-Layer LSTM (32-64 units) + Degree-Day Baseline  
**Owner:** Person 2 (Warisha) / ML Pipeline

### Key Invariants:
1. **Real Meteorological Forcing:** Temperature, snow depth, snowfall, freezing level height, and liquid precipitation from Open-Meteo ERA5.
2. **Deterministic Baseline:** Standardized degree-day melt and rain-on-snow heuristic (`DegreeDayBaseline`).
3. **Out-of-Sample Case Studies:** 2010 and 2022 historical cases evaluated out-of-sample against official hydrometric records (focusing on snow-fed basins: Swat Kalam, Chitral Mastuj, Chitral Drosh, Chitral town).

In [ ]:
# Setup environment and dependencies (Auto-detects Google Colab)
import os
import sys
from pathlib import Path

# If running in Google Colab, clone repo & install dependencies
if "google.colab" in sys.modules:
    if not (Path.cwd() / "src" / "floodwatch").exists():
        if not Path("FLASH_FLOOD_PREDICTION").exists():
            !git clone -b qamran/real-data-eval https://github.com/Syed-Muhammad-Qamran/FLASH_FLOOD_PREDICTION.git
        %cd FLASH_FLOOD_PREDICTION
    !pip install -q pyyaml pyarrow

repo_root = Path.cwd()
if not (repo_root / "src" / "floodwatch").exists() and (repo_root.parent / "src" / "floodwatch").exists():
    repo_root = repo_root.parent

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository Root: {repo_root}")
print(f"Computation Device: {device}")

In [ ]:
from floodwatch.features import load_basins, add_features, validate_no_leakage
from floodwatch.data_sources import openmeteo, cache
from floodwatch.models.model_b import build_model, build_baseline, TrainConfig

config_path = repo_root / "configs" / "real_data_training.yaml"
cfg = yaml.safe_load(config_path.read_text(encoding="utf-8"))
m_cfg = cfg["model_b"]
feature_cols = m_cfg["features"]
lookback = m_cfg["lookback_h"]
horizon = m_cfg["label_horizon_h"]

basins = load_basins()
print(f"Model B Features: {feature_cols}")

In [ ]:
# Ingest ERA5 Weather with Bounded Backoff and Caching
cache_base = repo_root / "data" / "raw" / "cache"
cache_base.mkdir(parents=True, exist_ok=True)

# Increased dataset including the 2010 Super Floods event (2010 through 2021)
START_DATE = "2010-06-01"
END_DATE = "2021-12-31"

basin_dfs = []
for b in basins:
    res = openmeteo.fetch_archive(
        basin_id=b["basin_id"],
        latitude=b["lat"],
        longitude=b["lon"],
        start_date=START_DATE,
        end_date=END_DATE,
        elevation_m=b.get("elevation_mean_m"),
        terrain_susceptibility=b.get("terrain_susceptibility"),
        snow_fraction=b.get("snow_fraction"),
        use_cache=True,
        cache_dir=cache_base,
    )
    basin_dfs.append(res.frame)

raw_df = pd.concat(basin_dfs, ignore_index=True)
raw_df["timestamp_utc"] = pd.to_datetime(raw_df["timestamp_utc"], utc=True)
print(f"Ingested {len(raw_df):,} hourly records.")

In [ ]:
# Build Melt & Rain-on-Snow Features
feat_df = add_features(raw_df)
validate_no_leakage(feat_df, feature_cols)

# Runoff proxy trigger: high degree-day melt OR liquid rain on deep snowpack
is_melt_trigger = (
    (feat_df["degree_day_melt"].fillna(0) >= 0.8)
    | (feat_df["rain_on_snow"].fillna(0) >= 6.0)
)

grouped = feat_df.groupby("basin_id", group_keys=False, sort=False)
feat_df["label"] = grouped.apply(
    lambda g: is_melt_trigger.loc[g.index]
    .iloc[::-1]
    .rolling(horizon, min_periods=1)
    .max()
    .iloc[::-1]
    .shift(-horizon)
    .fillna(0)
    .astype(int)
)

print(f"Melt/Rain-on-snow proxy labels: {feat_df['label'].sum():,} hours ({feat_df['label'].mean():.2%})")

In [ ]:
# Split and Preprocess
ts = feat_df["timestamp_utc"]
buffer_cutoff = pd.to_datetime("2020-12-01T00:00:00Z")
val_cutoff = pd.to_datetime("2021-01-01T00:00:00Z")

feat_df["split"] = "train"
feat_df.loc[(ts >= buffer_cutoff) & (ts < val_cutoff), "split"] = "buffer"
feat_df.loc[ts >= val_cutoff, "split"] = "validation"

validate_no_leakage(feat_df, feature_cols)

from sklearn.preprocessing import StandardScaler
train_mask = feat_df["split"] == "train"
val_mask = feat_df["split"] == "validation"

scaler = StandardScaler()
scaler.fit(feat_df.loc[train_mask, feature_cols])

scaled_df = feat_df.copy()
scaled_df[feature_cols] = scaler.transform(feat_df[feature_cols]).fillna(0.0)

# Build baseline on train rows
baseline_fn, baseline_state = build_baseline(feat_df[train_mask])
print(f"Degree-Day Baseline Fitted Intercept: {baseline_state.intercept:.3f}")

In [ ]:
# Train Model B LSTM
from torch.utils.data import Dataset, DataLoader

class TemporalDatasetB(Dataset):
    def __init__(self, sequences, idx_df, lookback=72):
        self.sequences = {k: torch.tensor(v, dtype=torch.float32) for k, v in sequences.items()}
        self.idx_df = idx_df.reset_index(drop=True)
        self.lookback = lookback
        self.b_ids = self.idx_df["basin_id"].values
        self.steps = self.idx_df["step_idx"].values
        self.targets = torch.tensor(self.idx_df["target"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.idx_df)

    def __getitem__(self, i):
        b, s = self.b_ids[i], self.steps[i]
        x_seq = self.sequences[b][s - self.lookback : s]
        return x_seq, self.targets[i]

basin_arrays = {b_id: grp[feature_cols].values.astype(np.float32) for b_id, grp in scaled_df.groupby("basin_id")}
index_records = []
for b_id, grp in scaled_df.groupby("basin_id"):
    g = grp.sort_values("timestamp_utc").reset_index(drop=True)
    times, splits, lbls = g["timestamp_utc"].values, g["split"].values, g["label"].values.astype(np.float32)
    for step in range(lookback, len(g)):
        if splits[step] == "buffer":
            continue
        index_records.append({"basin_id": b_id, "timestamp_utc": times[step], "step_idx": step, "split": splits[step], "target": lbls[step]})

index_df = pd.DataFrame(index_records)
train_idx = index_df[index_df["split"] == "train"].reset_index(drop=True)
val_idx = index_df[index_df["split"] == "validation"].reset_index(drop=True)

train_loader = DataLoader(TemporalDatasetB(basin_arrays, train_idx, lookback), batch_size=256, shuffle=True)
val_loader = DataLoader(TemporalDatasetB(basin_arrays, val_idx, lookback), batch_size=256, shuffle=False)

t_cfg = TrainConfig(seed=SEED, hidden_units=48, max_epochs=10)
model_b = build_model(n_features=len(feature_cols), cfg=t_cfg).to(device)
optimizer = torch.optim.Adam(model_b.parameters(), lr=1e-3)

print("Training Model B LSTM (1 layer, 48 units)...")
model_b.train()
for ep in range(1, 11):
    total_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy(model_b(x_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    print(f"Epoch {ep:02d} - Loss: {total_loss / len(train_idx):.4f}")

In [ ]:
# Validation Evaluation
model_b.eval()
val_preds = []
with torch.no_grad():
    for x_b, _ in val_loader:
        val_preds.append(model_b(x_b.to(device)).detach().cpu().numpy())
val_probs = np.concatenate(val_preds)
val_pred_binary = (val_probs >= 0.50).astype(int)
y_val = val_idx["target"].values

val_pod = float(np.sum((val_pred_binary == 1) & (y_val == 1)) / max(1, np.sum(y_val == 1)))
val_far = float(np.sum((val_pred_binary == 1) & (y_val == 0)) / max(1, np.sum(val_pred_binary == 1)))
print(f"Model B Validation Metrics: POD={val_pod:.4f} | FAR={val_far:.4f}")

In [ ]:
# External Historical Case Evaluation for Model B
from floodwatch.models.eval_cases import evaluate_case

for year, start_w, end_w in [(2010, "2010-07-28", "2010-08-03"), (2022, "2022-08-25", "2022-08-30")]:
    case_dfs = []
    for b in basins:
        res = openmeteo.fetch_archive(
            basin_id=b["basin_id"],
            latitude=b["lat"],
            longitude=b["lon"],
            start_date=start_w,
            end_date=end_w,
            use_cache=True,
            cache_dir=cache_base,
        )
        case_dfs.append(res.frame)
    c_raw = pd.concat(case_dfs, ignore_index=True)
    c_feats = add_features(c_raw)
    c_feats[feature_cols] = scaler.transform(c_feats[feature_cols]).fillna(0.0)

    case_basin_arrays = {b_id: grp[feature_cols].values.astype(np.float32) for b_id, grp in c_feats.groupby("basin_id")}
    case_records = []
    for b_id, grp in c_feats.groupby("basin_id"):
        g = grp.sort_values("timestamp_utc").reset_index(drop=True)
        times = g["timestamp_utc"].values
        for step in range(len(g)):
            case_records.append({"basin_id": b_id, "timestamp_utc": times[step], "step_idx": step})

    c_idx_df = pd.DataFrame(case_records)
    preds_case = []
    for row in c_idx_df.itertuples():
        arr = case_basin_arrays[row.basin_id]
        s = row.step_idx
        if s < lookback:
            pad = np.repeat(arr[:1], lookback - s, axis=0)
            seq = np.vstack([pad, arr[:s]])
        else:
            seq = arr[s - lookback : s]
        t_seq = torch.tensor(seq[np.newaxis, ...], dtype=torch.float32).to(device)
        with torch.no_grad():
            preds_case.append(float(model_b(t_seq).detach().cpu().item()))

    c_idx_df["prob"] = preds_case
    report = evaluate_case(case_year=year, predictions_df=c_idx_df, model_name="Model B", threshold=0.50)

    print(f"\n=== CASE {year}: {report.title} (Model B Melt Evaluation) ===")
    rows = []
    for c in report.comparisons:
        rows.append({
            "Basin": c.basin_id,
            "Confirmed Impact": c.confirmed_status,
            "Granularity": c.spatial_granularity,
            "Model Alert": c.model_predicted_risk,
            "Max Prob": c.model_max_prob,
            "Outcome": c.comparison_outcome,
        })
    display(pd.DataFrame(rows))

In [ ]:
# Save Model B Artifacts
out_dir = repo_root / "artifacts" / "b"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model_b.state_dict(), out_dir / "model_b_real.pt")

import json
manifest = {
    "module": "B_snowmelt_lstm",
    "seed": SEED,
    "features": feature_cols,
    "validation_pod": val_pod,
    "validation_far": val_far,
    "disclaimer": "Trained on real Open-Meteo ERA5 reanalysis for snowmelt/rain-on-snow flood risk.",
}
(out_dir / "manifest_real.json").write_text(json.dumps(manifest, indent=2))
print("Model B artifacts exported successfully.")